# Phase 6 — Calibration + Evaluation  (completes the core pipeline)

**Why (paper §3.8, §3.11):** the trained model's three outputs are *ordered* but not yet
*guaranteed* to contain the truth 90% of the time. **Conformal prediction** earns that
guarantee using the held-out calibration set:

1. On calibration, measure how far each truth fell outside the predicted range.
2. Take the 90% mark of those misses → a single number **Q**.
3. Widen every interval by **Q** (clip the lower end at 0).

Then we report the full metrics — all in **AQI points** — and, if you trained more than one
split strategy, compare them side by side to expose the leakage gap.

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Bootstrap — RUN ME FIRST ===
REPO_URL = "https://github.com/keyaan01/pm25-visual-aq.git"   # <-- your repo

import os, sys, shutil, subprocess

# Where the clone lives: Kaggle -> /kaggle/working ; Colab/local -> current dir.
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
ON_KAGGLE = os.path.isdir("/kaggle/working")

def _valid_repo(p):   # a REAL checkout of THIS repo, not a rogue/partial `src` left in the workdir
    return (os.path.exists(os.path.join(p, "src", "ceiling.py"))
            and os.path.exists(os.path.join(p, "configs", "default.yaml")))

if not ON_KAGGLE and _valid_repo("."):   # local/Colab dev already inside the repo -> use it as-is
    REPO = os.path.abspath(".")
else:                                    # Kaggle (or not in a repo): ALWAYS re-clone fresh
    REPO = os.path.join(BASE, "pm25-visual-aq")
    os.chdir(BASE)                       # don't stand inside the dir we're about to delete
    shutil.rmtree(REPO, ignore_errors=True)   # kill any stale / partial / rogue clone
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)

os.chdir(REPO)
sys.path.insert(0, REPO)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]                  # evict any `src` already imported from a stale location
# Fail LOUD if the checkout is incomplete (never silently import a namespace-package `src`):
assert os.path.exists(os.path.join(REPO, "src", "ceiling.py")), "clone incomplete (is Internet On?) -- src/ceiling.py missing at " + REPO

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

_head = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO, "| HEAD:", _head, "| Colab:", IN_COLAB)

In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt
from src.config import load_config
from src import data, splits, physics, dataset as D, model as M, train as T
from src import calibrate as C, metrics as Mx
cfg = load_config()
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

SOURCE = cfg["data"]["drive_path"]        # laptop test: "tests/fixture_ds"
ds, df = data.load_clean(SOURCE, from_disk=True, seed=cfg["seed"])
cache = physics.load_map_cache(os.path.join(cfg["data"]["cache_dir"], "physics_maps_%d.npy" % cfg["data"]["image_size"]))
out_root = cfg["data"]["outputs_dir"]; os.makedirs(out_root, exist_ok=True)
print("device:", device)

## Evaluate a trained model

`evaluate_strategy` rebuilds that split, loads its checkpoint, and predicts on calibration +
test. **Accuracy** (MAE/RMSE/R²) comes from the **point head**, de-standardised from its z-score
output back to AQI (it is trained with MSE, so it estimates the conditional mean — what R² rewards);
the **interval** (coverage/width) comes from the conformal-calibrated quantiles. It saves **Q** and
the point head's mean/std next to the checkpoint so the demo (Phase 10) can reuse them.

In [ ]:
import numpy as np
def evaluate_strategy(strategy):
    sp = splits.make_splits(df, strategy=strategy, seed=cfg["seed"],
            station_col=cfg["data"]["station_col"], time_col=cfg["data"]["time_col"],
            lon_col=cfg["data"]["lon_col"], lat_col=cfg["data"]["lat_col"])
    loaders = D.make_dataloaders(ds, sp, cache, cfg, num_workers=2)
    net = M.build_model(cfg).to(device)
    T.load_checkpoint(os.path.join(out_root, strategy, "best_model.pth"), net, map_location=device)
    cal = T.collect_outputs(net, loaders["cal"], device)
    test = T.collect_outputs(net, loaders["test"], device)
    # accuracy: de-standardised point head
    ymean, ystd = float(net.y_mean), float(net.y_std)
    point = T.point_to_aqi(test["point_out"], ymean, ystd)
    # intervals: conformal-calibrated quantiles
    Q = C.conformal_Q(np.exp(cal["q_log"]), cal["y_raw"], cfg["calibration"]["coverage"])
    intervals = C.apply_conformal(np.exp(test["q_log"]), Q)
    y = test["y_raw"]
    json.dump({"Q": Q, "y_mean": ymean, "y_std": ystd},
              open(os.path.join(out_root, strategy, "conformal_Q.json"), "w"))
    return {"strategy": strategy, "Q": Q, "point": point,
            "intervals": intervals, "y": y,
            "raw_coverage": C.coverage(np.exp(test["q_log"]), y),
            "report": Mx.report(point, intervals, y, cfg["calibration"]["coverage"])}

primary = evaluate_strategy(cfg["split"]["strategy"])
print("strategy:", primary["strategy"], "| Q = %.1f AQI" % primary["Q"])
print("coverage: raw %.3f -> calibrated %.3f (target %.2f)"
      % (primary["raw_coverage"], primary["report"]["coverage"], cfg["calibration"]["coverage"]))
{k: round(v,3) for k,v in primary["report"].items()}

## Where is the model weak?
Errors (of the point estimate) broken down by true-AQI band — watch the high-AQI bands, which Phase 5b targets.

In [ ]:
ebm = Mx.error_by_magnitude(primary["y"], primary["point"])
display(ebm)
plt.figure(figsize=(7,3)); plt.bar(ebm["band"], ebm["MAE"]); plt.ylabel("MAE (AQI)"); plt.xlabel("true AQI band")
plt.title("Error by pollution level"); plt.show()

w = primary["intervals"][:,2]-primary["intervals"][:,0]
plt.figure(figsize=(7,3)); plt.hist(w, bins=40)
plt.xlabel("interval width (AQI)"); plt.ylabel("count"); plt.title("Calibrated interval widths"); plt.show()

## A few example predictions

Each column is a test photo: the dot is the point estimate, the bar is the calibrated 90%
interval, and the ✕ is the truth. Most truths should sit inside their bars.

In [ ]:
idx = np.argsort(primary["y"])[::max(1, len(primary["y"])//40)][:40]
iv, pt, y = primary["intervals"][idx], primary["point"][idx], primary["y"][idx]
xs = np.arange(len(idx))
plt.figure(figsize=(9,4))
plt.vlines(xs, iv[:,0], iv[:,2], color="#4C78A8", lw=3, alpha=0.5, label="90% interval")
plt.plot(xs, pt, "o", ms=4, color="#4C78A8", label="point estimate")
plt.plot(xs, y, "x", ms=6, color="#E45756", label="truth")
plt.legend(); plt.xlabel("test photos (sorted by true AQI)"); plt.ylabel("AQI"); plt.title("Predictions vs truth"); plt.show()

## Leakage gap: compare every split you trained

If you trained more than one split strategy (e.g. re-ran `05_train` with
`split.strategy: random`), this tabulates them side by side. **`random` should look better
than `station_grouped`** — that difference is leakage inflation, measured not assumed.

In [ ]:
import pandas as pd
avail = [s for s in ["station_grouped","random","geographic","temporal"]
         if os.path.exists(os.path.join(out_root, s, "best_model.pth"))]
rows = []
for s in avail:
    r = evaluate_strategy(s)
    rows.append({"strategy": s, **{k: round(v,3) for k,v in r["report"].items()}})
table = pd.DataFrame(rows).set_index("strategy")
table.to_csv(os.path.join(out_root, "results_by_split.csv"))
table

## Core pipeline complete ✓

You now have: leakage-safe evaluation, a physics-guided model, and **calibrated intervals
with ~90% coverage**, all in AQI points. Paste me this notebook's numbers and I'll sanity-check
them and write them into `docs/RESULTS.md`.

**Next (contributions):** `07_error_ceiling` (C2 — how much error is unavoidable) and
`08_abstention` (C3 — refusing to answer on unusable inputs).